# Hybrid Agent: Qwen3-1.7B QLoRA on Google Colab

This easy-mode notebook verifies frozen inputs, trains a QLoRA adapter automatically, and exports the adapter plus reproducibility metadata. Optional switches enable base and adapter response generation. No human scoring is required. The current 12-record dataset validates the pipeline but is too small for a production-quality adapter.

## Setup

In Colab, select **Runtime → Change runtime type → GPU** before continuing. Set `REPO_URL` if the project is published; otherwise leave it blank and upload `hybrid-agent-colab-input.zip` when prompted.

In [ ]:
REPO_URL = ""  # Example: https://github.com/OWNER/REPOSITORY.git
BRANCH = "main"
# Easy mode: choose a GPU, Run all, upload the input ZIP, and wait for the download.
RUN_BASELINE = True  # Automatically score the pinned base model.
RUN_TRAINING = True
RUN_ADAPTER_EVALUATION = True  # Automatically score the trained adapter.
MINIMUM_MEANINGFUL_RECORDS = 3000
MAX_NEW_TOKENS = 384


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile

ROOT = Path('/content/hybrid-agent')
if ROOT.exists():
    shutil.rmtree(ROOT)
if REPO_URL:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, str(ROOT)], check=True)
else:
    from google.colab import files
    uploaded = files.upload()
    bundle = next((name for name in uploaded if name.endswith('.zip')), None)
    if bundle is None:
        raise RuntimeError('Upload hybrid-agent-colab-input.zip')
    ROOT.mkdir(parents=True)
    with zipfile.ZipFile(bundle) as archive:
        archive.extractall(ROOT)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('Project loaded at', ROOT)


In [ ]:
# Colab already supplies a CUDA-enabled PyTorch build. Install the remaining pinned stack.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'training/requirements-qlora-v1.txt'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. Enable a GPU runtime in Colab.')
gpu_name = torch.cuda.get_device_name(0)
compute_capability = torch.cuda.get_device_capability(0)
compute_dtype = torch.bfloat16 if compute_capability[0] >= 8 else torch.float16
print({'gpu': gpu_name, 'torch': torch.__version__, 'compute_dtype': str(compute_dtype)})


## Checks

Refuse changed or missing data before downloading the model.

In [ ]:
from hybrid_agent.evaluation import evaluate_responses
from hybrid_agent.experiments import verify_experiment
CONFIG_PATH = ROOT / 'training/configs/qwen3-1.7b-capability-v1.toml'
verified = verify_experiment(CONFIG_PATH, workspace=ROOT)
config = verified.config
print('Verified experiment:', config['experiment_id'])
print('Dataset manifest:', verified.manifest_sha256)
print('Evaluation suite:', verified.evaluation_sha256)


In [ ]:
import json, platform, time
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]

train_records = read_jsonl(verified.manifest_path.parent / 'train.jsonl')
validation_records = read_jsonl(verified.manifest_path.parent / 'validation.jsonl')
evaluation_cases = read_jsonl(verified.evaluation_path)
accepted_records = len(train_records) + len(validation_records)
run_class = 'candidate' if accepted_records >= MINIMUM_MEANINGFUL_RECORDS else 'pipeline-only'
print({'accepted_records': accepted_records, 'run_class': run_class})
if run_class == 'pipeline-only': print('WARNING: dataset is below the meaningful-run minimum; this adapter cannot be registered.')
run_dir = ROOT / config['output_dir']
run_dir.mkdir(parents=True, exist_ok=True)
tokenizer = AutoTokenizer.from_pretrained(config['model_id'], revision=config['model_revision'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype)
model = AutoModelForCausalLM.from_pretrained(config['model_id'], revision=config['model_revision'], quantization_config=quantization, device_map='auto', torch_dtype=compute_dtype)
model.config.use_cache = True
print('Loaded pinned base model')


## Steps

### 1. Capture the base-model responses

In [ ]:
def generate_response(active_model, prompt):
    messages = [{'role': 'user', 'content': prompt}]
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(rendered, return_tensors='pt').to(active_model.device)
    with torch.inference_mode():
        output = active_model.generate(**inputs, do_sample=False, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(output[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

base_responses = []
if RUN_BASELINE:
    for case in evaluation_cases:
        response = generate_response(model, case['prompt'])
        base_responses.append({'id': case['id'], 'prompt': case['prompt'], 'response': response})
        print(f"[{case['id']}] {response[:300]}\n")
with open(run_dir / 'base-responses.jsonl', 'w', encoding='utf-8') as stream:
    for row in base_responses: stream.write(json.dumps(row, ensure_ascii=False) + '\n')


### 2. Train the QLoRA adapter

This run follows the checked-in configuration. On GPUs without BF16 support, it records and uses FP16.

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

def render_record(record):
    return tokenizer.apply_chat_template(record['messages'], tokenize=False, add_generation_prompt=False, enable_thinking=False)

train_dataset = Dataset.from_dict({'text': [render_record(row) for row in train_records]})
validation_dataset = Dataset.from_dict({'text': [render_record(row) for row in validation_records]})
lora = config['lora']; training = config['training']
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=training['gradient_checkpointing'])
lora_config = LoraConfig(r=lora['r'], lora_alpha=lora['alpha'], lora_dropout=lora['dropout'], bias=lora['bias'], target_modules=lora['target_modules'], task_type='CAUSAL_LM')
sft_config = SFTConfig(output_dir=str(run_dir), seed=config['seed'], max_length=training['max_sequence_length'], num_train_epochs=training['epochs'], learning_rate=training['learning_rate'], lr_scheduler_type=training['lr_scheduler'], warmup_ratio=training['warmup_ratio'], per_device_train_batch_size=training['per_device_train_batch_size'], per_device_eval_batch_size=training['per_device_eval_batch_size'], gradient_accumulation_steps=training['gradient_accumulation_steps'], gradient_checkpointing=training['gradient_checkpointing'], optim=training['optimizer'], weight_decay=training['weight_decay'], max_grad_norm=training['max_grad_norm'], logging_steps=training['logging_steps'], eval_strategy='steps', eval_steps=training['evaluation_steps'], save_steps=training['save_steps'], save_total_limit=training['save_total_limit'], fp16=compute_dtype == torch.float16, bf16=compute_dtype == torch.bfloat16, report_to='none', dataset_text_field='text')
trainer = SFTTrainer(model=model, args=sft_config, train_dataset=train_dataset, eval_dataset=validation_dataset, processing_class=tokenizer, peft_config=lora_config)
train_metrics = None
if RUN_TRAINING:
    train_result = trainer.train()
    train_metrics = train_result.metrics
    trainer.save_model(run_dir / 'adapter')
    tokenizer.save_pretrained(run_dir / 'adapter')
    print(train_metrics)


### 3. Evaluate the adapter on the identical held-out suite

In [ ]:
model.config.use_cache = True
adapter_responses = []
if RUN_TRAINING and RUN_ADAPTER_EVALUATION:
    for case in evaluation_cases:
        response = generate_response(model, case['prompt'])
        adapter_responses.append({'id': case['id'], 'prompt': case['prompt'], 'response': response})
        print(f"[{case['id']}] {response[:300]}\n")
with open(run_dir / 'adapter-responses.jsonl', 'w', encoding='utf-8') as stream:
    for row in adapter_responses: stream.write(json.dumps(row, ensure_ascii=False) + '\n')


## Results

Responses are scored automatically with the frozen machine-readable assertions. No manual scoring or input is required.

In [ ]:
base_result = evaluate_responses(evaluation_cases, base_responses)
adapter_result = evaluate_responses(evaluation_cases, adapter_responses)
print('Base:', base_result)
print('Adapter:', adapter_result)


## Next Steps

Write the run manifest, hash the adapter, bundle all non-secret results, and download them before the Colab runtime disappears.

In [ ]:
import hashlib
def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
adapter_hashes = {}
adapter_dir = run_dir / 'adapter'
if adapter_dir.exists():
    adapter_hashes = {str(path.relative_to(run_dir)): file_hash(path) for path in sorted(adapter_dir.rglob('*')) if path.is_file()}
registration_eligible = run_class == 'candidate' and not adapter_result['critical_failures'] and adapter_result['score'] > base_result['score']
manifest = {'schema_version': 1, 'experiment_id': config['experiment_id'], 'status': 'trained' if RUN_TRAINING else 'setup-only', 'run_class': run_class, 'accepted_records': accepted_records, 'minimum_meaningful_records': MINIMUM_MEANINGFUL_RECORDS, 'registration_eligible': registration_eligible, 'model_id': config['model_id'], 'model_revision': config['model_revision'], 'dataset_manifest_sha256': verified.manifest_sha256, 'evaluation_suite_sha256': verified.evaluation_sha256, 'gpu': gpu_name, 'compute_dtype': str(compute_dtype), 'python': platform.python_version(), 'torch': torch.__version__, 'human_evaluation_required': False, 'base_score': base_result, 'adapter_score': adapter_result, 'training_metrics': train_metrics, 'adapter_sha256': adapter_hashes}
(run_dir / 'run-manifest.json').write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')
archive = shutil.make_archive('/content/qwen3-1.7b-capability-v1-results', 'zip', root_dir=run_dir)
print('Saved', archive)
from google.colab import files
files.download(archive)
